# Testing scipy linear programming

## Imports and definitions

In [28]:
# Module imports

import numpy as np

### Behaviors as vectors and notation

Suppose a (2,2,2) routed Bell experiment.

$ p(ab|xyz) \in \mathbb{R}^{32}$ is the observed behavior, assumed to be no-signaling : $$p \in \mathcal{NS}$$
We give ourselves notation for such behaviors, and a naïve generator.

We wish for easy conversion from a vector format (for algebraic operations) to a matrix format (for leggibility) :
$$p=\begin{pmatrix} p_{00|00S} \\ p_{00|01S} \\ \vdots \\ p_{00|00L} \\ \vdots \end{pmatrix}\quad \leftrightarrow \quad p=\begin{pmatrix} p_{00|00}& p_{00|01}& p_{00|10}& p_{00|11}\\ p_{01|00}& p_{01|01}& \dots\\ \vdots & & \ddots \\ & & & p_{11|11} \end{pmatrix}$$
where, to account for both matrices $p(z=S)$ and $p(z=L)$, both cases are represented in two matrices, making $p$ either a column vector in $\mathbb{R}^32$ or a third-order tensor of shape $(4,4,2)$.

Let us define hereunder some notable or useful such vectors, and utility functions :

In [ ]:
# TODO: Encapsulate the following in a dedicated behavior class


# The maximally mixed state over the experiment space
I = (1/4) * np.ones(32)

# The usual (2,2,2) PR box
SR_pr_box = np.array([1/2, 1/2, 1/2, 0,
                     0, 0, 0, 1/2,
                     0, 0, 0, 1/2,
                     1/2, 1/2, 1/2, 0,])

# The PR box in the experiment space : p(ab|xy) is assumed to be
# independent of the value of z
pr_box = np.concatenate((SR_pr_box, SR_pr_box), axis=0)

# Convert a vector behavior to its matrix representation
def behavior_vector_to_matrix(behavior_vector):
    """
    Convert a vector behavior to its matrix representation
    :param behavior_vector: The behavior vector
    :return: The behavior matrix
    """
    return np.reshape(behavior_vector, (2, 4, 4))

# ... and vice versa
def behavior_matrix_to_vector(behavior_matrix):
    """
    Convert a matrix behavior to its vector representation
    :param behavior_matrix: The behavior matrix
    :return: The behavior vector
    """
    return np.reshape(behavior_matrix, (32,))

# Given a 1-D index, return the corresponding a,b,x,y,z indices
def index_to_indices(index):
    """
    Given a 1-D index, return the corresponding a,b,x,y,z indices
    :param index: The 1-D index
    :return: The corresponding a,b,x,y,z indices
    """
    a = (index // 8) % 2
    b = (index // 4) % 2
    x = (index // 2) % 2
    y = index % 2
    z = index // 16
    return a, b, x, y, z

# Given a set of indices, return the corresponding 1-D index
def indices_to_index(a, b, x, y, z):
    """
    Given a set of indices, return the corresponding 1-D index
    :return: The corresponding 1-D index
    """
    return (z * 16) + (a * 8) + (b * 4) + (x * 2) + y

# Convert vector to matrix coordinates
def vector_to_matrix_coordinates(vector):
    """
    Convert vector to matrix coordinates
    :param vector: The vector
    :return: The matrix coordinates
    """
    return np.array([index_to_indices(i) for i in range(len(vector))])

# Convert matrix coordinates to vector
def matrix_coordinates_to_vector(matrix_coordinates):
    """
    Convert matrix coordinates to vector
    :param matrix_coordinates: The matrix coordinates
    :return: The vector
    """
    return np.array([indices_to_index(a, b, x, y, z) for a, b, x, y, z in matrix_coordinates])

In [30]:
# Sanity check cell, don't mind me

print(I.shape, "\n", I)
print()
print(pr_box.shape, "\n", pr_box)
print()

mat_pr_box = behavior_vector_to_matrix(pr_box)
print(mat_pr_box.shape, "\n", mat_pr_box)
print()
print(behavior_matrix_to_vector(mat_pr_box).shape, "\n", behavior_matrix_to_vector(mat_pr_box))
print()

for i in range(32):
    a, b, x, y, z = index_to_indices(i)
    print(f"i: {i} -> a: {a}, b: {b}, x: {x}, y: {y}, z: {z}")
    print(f"a: {a}, b: {b}, x: {x}, y: {y}, z: {z} -> i: {indices_to_index(a, b, x, y, z)}")
    print(i == indices_to_index(a, b, x, y, z))
print()

(32,) 
 [0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25
 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25
 0.25 0.25 0.25 0.25]

(32,) 
 [0.5 0.5 0.5 0.  0.  0.  0.  0.5 0.  0.  0.  0.5 0.5 0.5 0.5 0.  0.5 0.5
 0.5 0.  0.  0.  0.  0.5 0.  0.  0.  0.5 0.5 0.5 0.5 0. ]

(2, 4, 4) 
 [[[0.5 0.5 0.5 0. ]
  [0.  0.  0.  0.5]
  [0.  0.  0.  0.5]
  [0.5 0.5 0.5 0. ]]

 [[0.5 0.5 0.5 0. ]
  [0.  0.  0.  0.5]
  [0.  0.  0.  0.5]
  [0.5 0.5 0.5 0. ]]]

(32,) 
 [0.5 0.5 0.5 0.  0.  0.  0.  0.5 0.  0.  0.  0.5 0.5 0.5 0.5 0.  0.5 0.5
 0.5 0.  0.  0.  0.  0.5 0.  0.  0.  0.5 0.5 0.5 0.5 0. ]

i: 0 -> a: 0, b: 0, x: 0, y: 0, z: 0
a: 0, b: 0, x: 0, y: 0, z: 0 -> i: 0
True
i: 1 -> a: 0, b: 0, x: 0, y: 1, z: 0
a: 0, b: 0, x: 0, y: 1, z: 0 -> i: 1
True
i: 2 -> a: 0, b: 0, x: 1, y: 0, z: 0
a: 0, b: 0, x: 1, y: 0, z: 0 -> i: 2
True
i: 3 -> a: 0, b: 0, x: 1, y: 1, z: 0
a: 0, b: 0, x: 1, y: 1, z: 0 -> i: 3
True
i: 4 -> a: 0, b: 1, x: 0, y: 0, z: 0
a: 0, b: 1, x: 0,